# 3D Gaussian splatting in the Jupyter web visualizer

This notebook shows how to load a **3D Gaussian splatting** scene (``.ply`` or ``.splat`` file), inspect the tensor point cloud, view it with the **Jupyter visualizer** ``open3d.web_visualizer.draw`` or native visualizer ``open3d.visualization.draw``, crop and transform the splats, and display them with a **PBR model** in the same viewer.

The 3DGS file is read with ``open3d.t.io.read_point_cloud``; Gaussian attributes
(positions, scales, rotations, SH coefficients, opacity) follow the usual 3DGS layout.
Use ``PointCloud.Translate`` / ``Rotate`` / ``Scale`` for splats so orientations and
spherical-harmonic bands stay consistent (see the docs for ``t.geometry.PointCloud``).

In [1]:
from pathlib import Path
import urllib.request

import numpy as np
import open3d as o3d
#from open3d.web_visualizer import draw   # Jupyter web visualizer
from open3d.visualization import draw     # Native visualizer
from open3d.visualization.rendering import MaterialRecord

# Sample 3DGS scene (Hugging Face — raw file URL for download).
# SPLAT_URL = "https://huggingface.co/cakewalk/splat-data/resolve/main/garden.splat"
# splat_path = Path("garden.splat")
# Or use an existing ply file from disk
splat_path = Path('/home/ssheorey/Documents/data/3dgs-models/garden-30K.ply')
PBR_MESH_URL = "https://github.com/KhronosGroup/glTF-Sample-Assets/raw/refs/heads/main/Models/ChairDamaskPurplegold/glTF-Binary/ChairDamaskPurplegold.glb"
pbr_mesh_path = Path("ChairDamaskPurplegold.glb")

if not splat_path.is_file():
    print(f"Downloading {SPLAT_URL!r} ...")
    urllib.request.urlretrieve(SPLAT_URL, splat_path)
if not pbr_mesh_path.is_file():
    print(f"Downloading {PBR_MESH_URL!r} ...")
    urllib.request.urlretrieve(PBR_MESH_URL, pbr_mesh_path)
print(f"Using splat file: {splat_path.resolve()}")

Using splat file: /home/ssheorey/Documents/data/3dgs-models/garden-30K.ply


## Load the splat scene

Tensor IO loads ``.splat`` / Gaussian ``.ply`` into ``open3d.t.geometry.PointCloud``.
Printing the object summarizes point count and attribute names (e.g. ``positions``,
``scale``, ``rot``, ``opacity``, ``f_dc``, ``f_rest`` when present).

In [2]:
pcd = o3d.t.io.read_point_cloud(str(splat_path))
print(pcd)

PointCloud on CPU:0 [5834784 points (Float32)].
Attributes: rot (dtype = Float32, shape = {5834784, 4}), opacity (dtype = Float32, shape = {5834784, 1}), f_rest (dtype = Float32, shape = {5834784, 15, 3}), scale (dtype = Float32, shape = {5834784, 3}), f_dc (dtype = Float32, shape = {5834784, 3}), normals (dtype = Float32, shape = {5834784, 3}).


## Display with the visualizer

In [3]:
from open3d.visualization.rendering import MaterialRecord
mat = MaterialRecord()
mat.shader = "gaussianSplat"
mat.gaussian_splat_min_alpha = 0.05   # filter translucent splats
draw([{"name": "garden", "geometry": pcd, "material": mat}], title="Garden (full)", width=800, height=600, show_skybox=False, near_plane=1.0)

[Open3D INFO] GaussianSplatOpenGLContext: Wayland session detected; using X11/GLX via XWayland for Filament compatibility.


## Crop

We can ``Crop()`` the center portion of the scene to focus on the able in the garden.

In [4]:
full_bb = pcd.get_axis_aligned_bounding_box()
mn = full_bb.min_bound
mx = full_bb.max_bound
center = (mn + mx) * 0.5
extent = mx - mn
print(f"{center=}, {extent=}")
# Keep the middle portion along each axis (adjust factor as needed).
half = extent * 0.07
crop_bb = o3d.t.geometry.AxisAlignedBoundingBox(-half, half)
pcd_cropped = pcd.crop(crop_bb)
print("Points after crop:", pcd_cropped.point.positions.shape[0])
draw([{"name": "garden", "geometry": pcd_cropped, "material": mat}], title="Garden (crop)", 
     width=800, height=600, show_skybox=False, near_plane=1.0)

center=[6.3410206 -12.3407955 5.5567226]
Tensor[shape={3}, stride={1}, Float32, CPU:0, 0xf21e030], extent=[102.26373 43.325138 59.35886]
Tensor[shape={3}, stride={1}, Float32, CPU:0, 0xf4d14f0]
Points after crop: 2920664


## Translate and Rotate

For Gaussian splat clouds, we can edit them with ``Translate`` / ``Rotate`` / ``Scale`` / ``Crop``. All properties including SH coefficients are transformed accordinagly. The more generic ``Transform()`` (e.g. skew) is not supported.

We will move the scene centroid to the origin, then rotate it to align the ground with the YZ plane. Then we can crop the scene to focus on the table in the center.

In [5]:
# Move the cloud so its centroid is at the origin. With relative=False, the
# argument is the desired new center (see t::geometry::PointCloud::Translate).
pcd_xform = pcd_cropped.translate(
    o3d.core.Tensor([0.0, 0.0, 0.0]), relative=False
)
# Rotate so the the scene is Y up
R = o3d.geometry.get_rotation_matrix_from_xyz((np.pi, 0, 0))
pcd_xform = pcd.rotate(o3d.core.Tensor(R), o3d.core.Tensor([0.0, 0.0, 0.0]))
draw([{"name": "garden", "geometry": pcd_xform, "material": mat}], title="Garden (transformed)", 
     width=800, height=600,  show_skybox=False, near_plane=1.0, show_axes=True)

## Add a PBR triangle mesh next to the splats

We will now place a BoomBox on the table in the garden. Pass a **list** of geometries to ``draw`` to render both together.

In [22]:
pbr_mesh_path = Path("/home/ssheorey/Downloads/ChairDamaskPurplegold.glb")
gltf = o3d.io.read_triangle_model(pbr_mesh_path)
#mesh_dict = o3d.t.geometry.TriangleMesh.from_triangle_mesh_model(gltf)
#print(mesh_dict)
#mesh = mesh_dict["BoomBox"]
#m_bb = mesh.get_axis_aligned_bounding_box()
#m_ctr = m_bb.get_center()
#mesh = mesh.scale(1.0, m_ctr)
#mesh = mesh.translate(o3d.core.Tensor([0.0, 0.0, 0.0]))
#draw(gltf)
draw([{"name": "garden", "geometry": pcd_xform, "material": mat}, {"name": "Chair", "geometry": gltf}], title="Garden splats + boombox mesh", width=900, height=650, near_plane=1.0)

In [8]:
o3d.t.geometry.TriangleMesh.from_triangle_mesh_model(mesh)

{'BoomBox': TriangleMesh on CPU:0 [3575 vertices (Float32) and 6036 triangles (Int64)].
 Vertex Attributes: normals (dtype = Float32, shape = {3575, 3}).
 Triangle Attributes: texture_uvs (dtype = Float32, shape = {6036, 3, 2}).}